# Quantum Support Vector Machine for Fraud Detection
**ArXivist-generated reproduction notebook**

> **Paper:** Quantum Support Vector Machine for Fraud Detection  
> **Authors:** Liang Ren, Xuesong Zhang  
> **Venue:** IEEE CCPQT 2025 | **DOI:** 10.1109/CCPQT66408.2025.11383198  
> **Generated by:** ArXivist v1.0 | **SIR Confidence:** 0.77

---

This notebook walks through every component of the implementation, runs a
**mini-training demonstration on synthetic data** (no downloads required), and
shows you exactly how to launch the full training run that reproduces the paper.

**Structure:**
1. Environment check
2. Installation
3. Paper overview
4. Component walkthroughs (data → SMOTE → feature map → kernel → QSVM)
5. Mini-training on synthetic data (runs in < 5 min)
6. Paper results comparison
7. Next steps

## 1. Environment Check

In [ ]:
import sys
print(f"Python: {sys.version}")

# Check Qiskit
try:
    import qiskit
    print(f"Qiskit: {qiskit.__version__} ✓")
except ImportError:
    print("Qiskit: NOT INSTALLED ✗  →  run: pip install qiskit qiskit-aer qiskit-machine-learning")

# Check Qiskit Aer
try:
    import qiskit_aer
    print(f"Qiskit-Aer: {qiskit_aer.__version__} ✓")
except ImportError:
    print("Qiskit-Aer: NOT INSTALLED ✗")

# Check qiskit-machine-learning
try:
    import qiskit_machine_learning
    print(f"qiskit-machine-learning: {qiskit_machine_learning.__version__} ✓")
except ImportError:
    print("qiskit-machine-learning: NOT INSTALLED ✗")

# Check sklearn
try:
    import sklearn
    print(f"scikit-learn: {sklearn.__version__} ✓")
except ImportError:
    print("scikit-learn: NOT INSTALLED ✗")

import numpy as np
print(f"NumPy: {np.__version__} ✓")
print()
print("Note: This paper uses CPU-only quantum simulation (no GPU/CUDA required).")

## 2. Installation

In [ ]:
import subprocess, sys

# Install the project in editable mode (run this once)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".."],
    capture_output=True, text=True, cwd=".."
)
if result.returncode == 0:
    print("Installation successful ✓")
    # Show key installed versions
    for pkg in ["qiskit", "qiskit_aer", "qiskit_machine_learning", "sklearn"]:
        try:
            mod = __import__(pkg)
            print(f"  {pkg}: {getattr(mod, '__version__', 'installed')}")
        except ImportError:
            print(f"  {pkg}: NOT FOUND")
else:
    print("Installation output:")
    print(result.stdout[-2000:])
    print("STDERR:", result.stderr[-1000:])

In [ ]:
# Add src/ to Python path (in case editable install isn't active yet)
import sys
from pathlib import Path

src_path = str(Path("..").resolve() / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

repo_root = str(Path("..").resolve())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"src path: {src_path}")

## 3. Paper Overview

### Problem
Credit card fraud detection faces two major challenges:
1. **Severe class imbalance** — only 0.172% of transactions are fraudulent (492 / 284,807)
2. **High-dimensional features** — 30 PCA-derived features that may not be linearly separable

### Core Innovation
The paper proposes a two-part quantum solution:

**Part 1 — Quantum-SMOTE** (Section II): Instead of classical KNN-based oversampling, it uses:
- K-means clustering of minority (fraud) samples
- Amplitude encoding of samples into quantum states
- Quantum swap test to compute angular distances
- Quantum Ry-rotation to synthesize new minority samples

**Part 2 — QSVM** (Section III): Classical SVM + quantum kernel:
$$K(x_i, x_j) = |\langle\phi(x_i)|\phi(x_j)\rangle|^2$$

The quantum feature map $U_\phi(x)$ (ZZFeatureMap) embeds data into a $2^n$-dimensional
Hilbert space, making previously inseparable patterns separable.

### Key Results (Table I & II)
| Config | Accuracy | F1 | Recall | AUC |
|--------|---------|-----|--------|-----|
| QSVM-10qubit + Quantum-SMOTE | **98.8%** | **0.962** | **0.945** | **0.992** |
| QSVM-10qubit + Undersampling | 92.4% | 0.907 | 0.881 | 0.972 |
| SVM-10feat (classical) | 95.1% | 0.933 | 0.923 | 0.979 |

### Implementation Map
```
Paper Section → Code Module
─────────────────────────────────────────────────────
Section II    → src/qsvm_fraud/models/quantum_smote.py
Section III-A → src/qsvm_fraud/models/feature_map.py
Section III-B → src/qsvm_fraud/models/quantum_kernel.py
Section III-C → src/qsvm_fraud/models/qsvm.py
Section IV-A  → src/qsvm_fraud/data/dataset.py
Section IV-B  → src/qsvm_fraud/evaluation/metrics.py
```

## 4. Component Walkthrough
### 4.1 Configuration

In [ ]:
from qsvm_fraud.utils.config import Config

# Load primary config (10-qubit + Quantum-SMOTE)
config = Config.load("../configs/config.yaml")

print("Model config:")
for k, v in config["model"].items():
    if not k.startswith("_"):
        print(f"  {k}: {v}")

print("\nData config:")
for k, v in config["data"].items():
    if not k.startswith("_"):
        print(f"  {k}: {v}")

print("\nQuantum-SMOTE config:")
for k, v in config["quantum_smote"].items():
    if not k.startswith("_"):
        print(f"  {k}: {v}")

# Set reproducibility seed
Config.set_seed(config["hardware"]["random_seed"])
print("\nSeed set ✓")

### 4.2 Quantum Feature Map (ZZFeatureMap)

The ZZFeatureMap encodes classical data $x \in \mathbb{R}^d$ into a quantum state:
$$|\phi(x)\rangle = U_\phi(x)|0\rangle^{\otimes n}$$

The circuit consists of Hadamard layers followed by ZZ entangling layers:
$$U_\phi(x) = \prod_{\text{rep}} \left( \prod_j e^{i x_j Z_j} \prod_{j<k} e^{i x_j x_k Z_j Z_k} H^{\otimes n} \right)$$

This embeds data into a $2^n = 2^{10} = 1024$-dimensional Hilbert space.

**Paper:** Section III-A, Eq. 3 | `models/feature_map.py`

> **ASSUMED:** `reps=2`, `entanglement='full'` (Qiskit defaults; confidence 0.60)

In [ ]:
from qsvm_fraud.models.feature_map import QSVMFeatureMap

# Build the ZZFeatureMap for 4 qubits (small for display)
fm_demo = QSVMFeatureMap(
    n_qubits=4,          # Paper: 4, 8, or 10 qubits (primary: 10)
    reps=2,              # ASSUMED: Qiskit default (confidence: 0.60)
    entanglement="full", # ASSUMED: Qiskit default (confidence: 0.60)
)

feature_map = fm_demo.build()
print(f"Feature map: {fm_demo}")
print(f"Number of qubits: {feature_map.num_qubits}")
print(f"Number of parameters: {feature_map.num_parameters}")
print(f"Circuit depth (approx): {feature_map.decompose().depth()}")

# Draw the circuit (text mode)
try:
    circuit = fm_demo.get_circuit()
    print("\nCircuit diagram (4-qubit ZZFeatureMap):")
    print(circuit.draw(output="text", fold=80))
except Exception as e:
    print(f"Circuit drawing skipped: {e}")

### 4.3 Quantum Kernel Function

The quantum kernel is defined as the fidelity between two feature states:

$$K(x_i, x_j) = |\langle\phi(x_i)|\phi(x_j)\rangle|^2$$

Computed via the **swap test** (Eqs. 5–7):
$$P_0 = \frac{1}{2}\left(1 + |\langle\phi(x_i)|\phi(x_j)\rangle|^2\right) \quad \Rightarrow \quad K(x_i, x_j) = 2P_0 - 1$$

In Qiskit, this is implemented as:
$$K_{ij} = |\langle 0^n | U_\phi(x_j)^\dagger U_\phi(x_i) | 0^n \rangle|^2$$

**Paper:** Section III-B, Eqs. 4–8 | `models/quantum_kernel.py`

In [ ]:
import numpy as np
from qsvm_fraud.models.feature_map import QSVMFeatureMap
from qsvm_fraud.models.quantum_kernel import QSVMKernelComputer

# Use 4-qubit feature map for speed
fm = QSVMFeatureMap(n_qubits=4, reps=2, entanglement="full")
kernel_computer = QSVMKernelComputer(
    feature_map=fm,
    backend="statevector_simulator",
    cache_dir="/tmp/kernel_cache_demo/",
)

# Compute kernel between two toy data points
np.random.seed(42)
xi = np.random.randn(4)  # 4-dimensional feature vector
xj = np.random.randn(4)

try:
    k_val = kernel_computer.kernel_entry(xi, xj)
    print(f"Kernel demo (2 random 4-dim vectors):")
    print(f"  xi = {xi.round(3)}")
    print(f"  xj = {xj.round(3)}")
    print(f"  K(xi, xj) = {k_val:.4f}  (range [0, 1])")

    # Self-kernel should be 1.0 (or very close)
    k_self = kernel_computer.kernel_entry(xi, xi)
    print(f"  K(xi, xi) = {k_self:.4f}  (expected: 1.0 — self-similarity)")
except Exception as e:
    print(f"Kernel computation error: {e}")
    print("Ensure qiskit-aer and qiskit-machine-learning are installed.")

### 4.4 Kernel Matrix Computation

The full training kernel matrix $K \in \mathbb{R}^{N \times N}$ (Eq. 8):
$$K_{ij} = K(x_i, x_j) = |\langle\phi(x_i)|\phi(x_j)\rangle|^2$$

This is passed to `SVC(kernel='precomputed')` for classical dual optimisation.

> ⚠️ **Computational note:** Computing the full $K$ matrix is $O(N^2)$ quantum circuit 
> evaluations. For 500 training samples that's 250,000 evaluations. The demo below uses 
> $N=10$ for speed.

In [ ]:
import numpy as np
import time
from qsvm_fraud.models.feature_map import QSVMFeatureMap
from qsvm_fraud.models.quantum_kernel import QSVMKernelComputer

# Tiny demo: 10 samples, 4 qubits
np.random.seed(42)
X_tiny = np.random.randn(10, 4)

fm = QSVMFeatureMap(n_qubits=4, reps=2, entanglement="full")
kc = QSVMKernelComputer(fm, cache_dir="/tmp/kernel_demo/")

try:
    t0 = time.time()
    K = kc.compute_kernel_matrix(X_tiny)
    elapsed = time.time() - t0

    print(f"Kernel matrix shape: {K.shape}  (10×10 for 10 training samples)")
    print(f"Computed in: {elapsed:.2f}s")
    print(f"Diagonal (self-similarity, should all be ~1.0):")
    print(f"  {np.diag(K).round(3)}")
    print(f"Kernel matrix range: [{K.min():.3f}, {K.max():.3f}]")
    print(f"Matrix is symmetric: {np.allclose(K, K.T, atol=1e-6)}")

    # Estimate full-dataset runtime
    n_train_full = 500  # conservative estimate
    per_pair = elapsed / (10 * 10)
    est_full = per_pair * n_train_full * n_train_full
    print(f"\nEstimated time for {n_train_full}×{n_train_full} kernel matrix:")
    print(f"  ~{est_full/60:.1f} minutes on this machine")
    print("  → Use cache_kernel=True (default) to avoid recomputation.")
except Exception as e:
    print(f"Kernel matrix error: {e}")

### 4.5 Quantum-SMOTE

Addresses the severe class imbalance (0.172% fraud) by synthesizing new minority-class samples:

1. **K-means clustering** — group minority samples into $K$ clusters
2. **Amplitude encoding** (Eq. 1): $x \mapsto |x\rangle = \sum_i \frac{x_i}{\|x\|}|i\rangle$
3. **Swap test** — compute angular distance between sample and cluster centroid
4. **Quantum rotation** (Ry gate) — synthesize new sample: $x_{\text{syn}} = x + \theta(c - x)$

**Paper:** Section II | `models/quantum_smote.py`

> ⚠️ **SIR Confidence: 0.70** — The exact rotation circuit is underspecified in the paper.
> The synthesis step uses the classical interpolation analogue pending verification against
> Mohanty et al. 2025.

In [ ]:
import numpy as np
from qsvm_fraud.models.quantum_smote import QuantumSMOTE, ClassicalSMOTE, build_smote

# Create synthetic imbalanced dataset (100 majority, 10 minority)
np.random.seed(42)
n_maj, n_min = 100, 10
X_maj = np.random.randn(n_maj, 4) + np.array([2, 2, 2, 2])   # majority cluster
X_min = np.random.randn(n_min, 4) + np.array([-2, -2, -2, -2]) # minority cluster
X_imb = np.vstack([X_maj, X_min])
y_imb = np.array([0]*n_maj + [1]*n_min)

print(f"Before Quantum-SMOTE:")
print(f"  Total samples: {len(X_imb)}")
print(f"  Majority (class 0): {(y_imb==0).sum()}")
print(f"  Minority (class 1): {(y_imb==1).sum()}")
print(f"  Imbalance ratio: {(y_imb==1).sum()/len(y_imb):.1%}")

try:
    smote = QuantumSMOTE(
        n_clusters=3,
        rotation_angle=0.5,   # ASSUMED (confidence: 0.45)
        minority_ratio=0.5,   # Target 50/50 balance
        segmentation_factor=1.0,
        random_state=42,
    )
    X_bal, y_bal = smote.fit_resample(X_imb, y_imb)

    print(f"\nAfter Quantum-SMOTE:")
    print(f"  Total samples: {len(X_bal)}")
    print(f"  Majority (class 0): {(y_bal==0).sum()}")
    print(f"  Minority (class 1): {(y_bal==1).sum()}")
    print(f"  Balance ratio: {(y_bal==1).sum()/len(y_bal):.1%}")
    print(f"  Synthetic samples generated: {(y_bal==1).sum() - n_min}")

except Exception as e:
    print(f"QuantumSMOTE error: {e}")
    print("Falling back to ClassicalSMOTE...")
    smote = ClassicalSMOTE(random_state=42)
    X_bal, y_bal = smote.fit_resample(X_imb, y_imb)
    print(f"ClassicalSMOTE result: {len(X_bal)} samples, ratio={y_bal.mean():.1%} fraud")

### 4.6 Full QSVM Model

The QSVM orchestrates: ZZFeatureMap → kernel matrix → classical SVC.

**Decision function (Eq. 9):**
$$f(x) = \text{sign}\left(\sum_{i=1}^{m} \alpha_i y_i K(x_i, x) + b\right)$$

where $\alpha_i$ are the dual coefficients and $b$ is the bias learned by `SVC(kernel='precomputed')`.

**Paper:** Section III-C | `models/qsvm.py`

In [ ]:
from qsvm_fraud.models.qsvm import QSVM
print("QSVM class loaded ✓")
print()

# Show what a QSVM instance looks like
qsvm = QSVM(
    n_qubits=4,            # 4-qubit demo (paper primary: 10)
    reps=2,                # ASSUMED (confidence: 0.60)
    entanglement="full",   # ASSUMED (confidence: 0.60)
    C=1.0,                 # ASSUMED (confidence: 0.65)
    backend="statevector_simulator",
    cache_kernel=False,    # No caching in demo
    probability=True,
    random_state=42,
)
print(f"QSVM instance: {qsvm}")
print()
print("Internal components:")
print(f"  Feature map: {qsvm._feature_map}")
print(f"  Kernel computer: {qsvm._kernel_computer}")
print(f"  SVC: {qsvm._svc}")

## 5. Mini-Training Demonstration

**Runs entirely on synthetic data — no download required.**

We train a 4-qubit QSVM on a small synthetic binary classification dataset
(80 samples, 4 features) — a miniature version of the fraud detection task.

This verifies that the full pipeline works end-to-end:
`data → SMOTE → scale → kernel → SVC → predict → metrics`

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from qsvm_fraud.models.quantum_smote import QuantumSMOTE, ClassicalSMOTE
from qsvm_fraud.models.qsvm import QSVM
from qsvm_fraud.evaluation.metrics import FraudMetrics

np.random.seed(42)

# ── Synthetic dataset (mimics credit card fraud imbalance) ──────────────────
n_legit, n_fraud = 80, 8
X_legit = np.random.randn(n_legit, 4) * 0.8 + np.array([1, 0, -1, 0.5])
X_fraud = np.random.randn(n_fraud,  4) * 0.5 + np.array([-1, 0.5, 1, -0.5])
X_raw = np.vstack([X_legit, X_fraud])
y_raw = np.array([0]*n_legit + [1]*n_fraud)

print(f"Synthetic dataset: {len(X_raw)} samples | {(y_raw==1).sum()} fraud ({y_raw.mean():.1%})")

# ── Train/test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.25, stratify=y_raw, random_state=42
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# ── Preprocessing ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Quantum-SMOTE oversampling ──────────────────────────────────────────────
print("\nApplying oversampling...")
try:
    smote = QuantumSMOTE(n_clusters=2, rotation_angle=0.5, minority_ratio=0.45, random_state=42)
    X_bal, y_bal = smote.fit_resample(X_train_sc, y_train)
    print(f"QuantumSMOTE → balanced: {len(X_bal)} samples | fraud: {y_bal.sum()}")
except Exception as e:
    print(f"QuantumSMOTE failed ({e}), using ClassicalSMOTE fallback")
    smote = ClassicalSMOTE(random_state=42)
    X_bal, y_bal = smote.fit_resample(X_train_sc, y_train)
    print(f"ClassicalSMOTE → balanced: {len(X_bal)} samples | fraud: {y_bal.sum()}")

# ── QSVM training ───────────────────────────────────────────────────────────
print("\nTraining 4-qubit QSVM...")
print("(Computing kernel matrix — this may take 1–2 minutes on CPU)")
import time
t0 = time.time()

qsvm = QSVM(
    n_qubits=4,
    C=1.0,
    backend="statevector_simulator",
    cache_kernel=False,
    probability=True,
    random_state=42,
)
try:
    qsvm.fit(X_bal, y_bal)
    elapsed = time.time() - t0
    print(f"QSVM trained in {elapsed:.1f}s ✓")
    print(f"Status: {qsvm}")
except Exception as e:
    print(f"QSVM training failed: {e}")
    qsvm = None

In [ ]:
# ── Evaluation ──────────────────────────────────────────────────────────────
if qsvm is not None:
    print("Evaluating on test set...")
    y_pred = qsvm.predict(X_test_sc)
    y_score = qsvm.predict_proba(X_test_sc)

    from qsvm_fraud.evaluation.metrics import FraudMetrics
    metrics_tool = FraudMetrics()
    result = metrics_tool.compute(y_test, y_pred, y_score, label="QSVM-4qubit-synthetic")
    metrics_tool.print_report(result)
else:
    print("QSVM not fitted — skipping evaluation.")

In [ ]:
# ── Classical SVM baseline for comparison ───────────────────────────────────
from sklearn.svm import SVC
from qsvm_fraud.evaluation.metrics import FraudMetrics

svc = SVC(kernel="rbf", C=1.0, probability=True, random_state=42)
svc.fit(X_train_sc, y_train)
y_pred_svm = svc.predict(X_test_sc)
y_score_svm = svc.predict_proba(X_test_sc)

metrics_tool = FraudMetrics()
result_svm = metrics_tool.compute(y_test, y_pred_svm, y_score_svm, label="SVM-RBF-4feat-baseline")
metrics_tool.print_report(result_svm)

## 6. Paper Results Comparison

The table below shows the results reported in the paper (from SIR `evaluation_protocol`).
After running the full training pipeline, compare your outputs to these targets.

In [ ]:
# Results reported in the paper (SIR evaluation_protocol.reported_results)
paper_results = {
    "Primary config": {
        "model": "QSVM-10qubit + Quantum-SMOTE",
        "accuracy":  98.8,
        "f1":        0.962,
        "recall":    0.945,
        "auc":       0.992,
    },
    "Undersampling baseline": {
        "model": "QSVM-10qubit + Undersampling",
        "accuracy":  92.4,
        "f1":        0.907,
        "recall":    0.881,
        "auc":       0.972,
    },
    "Classical SVM 10-feat": {
        "model": "SVM-10feat",
        "accuracy":  95.1,
        "f1":        0.933,
        "recall":    0.923,
        "auc":       0.979,
    },
    "QSVM 4-qubit": {"model": "QSVM-4qubit", "accuracy": 92.2, "f1": 0.902, "recall": 0.894, "auc": 0.971},
    "QSVM 8-qubit": {"model": "QSVM-8qubit", "accuracy": 96.7, "f1": 0.956, "recall": 0.941, "auc": 0.983},
}

print(f"{'Model':<40} {'Accuracy':>10} {'F1':>8} {'Recall':>8} {'AUC':>8}")
print("-" * 78)
for name, r in paper_results.items():
    print(f"{r['model']:<40} {r['accuracy']:>9.1f}% {r['f1']:>8.3f} {r['recall']:>8.3f} {r['auc']:>8.3f}")

print("\nDataset: Kaggle Credit Card Fraud Detection (284,807 transactions)")
print("Source: Tables I & II of Ren & Zhang, IEEE CCPQT 2025")
print()
print("SIR confidence on reported results: 0.88 (high)")
print("SIR confidence on implementation assumptions: 0.68 (medium)")

In [ ]:
# Key implementation assumptions with confidence scores
assumptions = [
    ("model.n_qubits", 10, 0.97, "Explicitly stated — primary config"),
    ("ZZFeatureMap.reps", 2, 0.60, "ASSUMED: Qiskit default"),
    ("ZZFeatureMap.entanglement", "full", 0.60, "ASSUMED: Qiskit default"),
    ("SVC.C", 1.0, 0.65, "ASSUMED: sklearn default — not stated in paper"),
    ("backend", "statevector_simulator", 0.85, "ASSUMED: ideal simulation"),
    ("KBest.score_func", "f_classif", 0.60, "ASSUMED: standard default"),
    ("train/test split", "80/20", 0.55, "ASSUMED — not stated in paper"),
    ("SMOTE rotation_angle", 0.5, 0.45, "ASSUMED — paper lists as hyperparameter, no value given"),
]

print("\nImplementation Assumptions (sorted by confidence):")
print(f"{'Parameter':<30} {'Value':<22} {'Conf':>6}  Notes")
print("-" * 80)
for param, val, conf, note in sorted(assumptions, key=lambda x: -x[2]):
    status = "✓" if conf >= 0.80 else ("⚠" if conf >= 0.60 else "✗")
    print(f"{status} {param:<28} {str(val):<22} {conf:>5.2f}  {note}")

print("\n✓ = high confidence  ⚠ = medium confidence  ✗ = low confidence (verify manually)")

## 7. Full Training Run

To reproduce the paper's results on the full Kaggle dataset:

### Step 1 — Get the dataset
```bash
# Download creditcard.csv (requires Kaggle account)
python ../scripts/download_data.py --output-dir ../data/raw/
# Or download manually from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
```

### Step 2 — Train (primary config: 10-qubit + Quantum-SMOTE)
```bash
cd ..  # repo root
python train.py --config configs/config.yaml --csv data/raw/creditcard.csv
```

### Step 3 — Run ablation study (Tables I & II)
```bash
python scripts/run_ablation.py --csv data/raw/creditcard.csv
```

### Step 4 — Evaluate saved model
```bash
python evaluate.py \
    --model-path checkpoints/qsvm_10qubit_qsmote.joblib \
    --csv data/raw/creditcard.csv \
    --save-plots
```

### ⚠️ Performance Warning
The quantum kernel matrix is **O(N²)**. For N=500 training samples with 4-qubit 
circuits, expect ~5–15 minutes. For larger N or 10-qubit circuits, this scales 
significantly. Use `--max-samples` to control training set size:

```bash
python train.py --config configs/config.yaml \
                --csv data/raw/creditcard.csv \
                --max-samples 200  # fast test
```

### Next Steps
1. Feed your results to **ArXivist Stage 6 (Results Comparator)** to automatically
   compare against the paper's reported values
2. If metrics diverge, check: SVM C value, ZZFeatureMap reps, train/test split ratio
3. See `README.md` for the full list of implementation assumptions to verify